# Volatility Model Benchmark

The point of this notebook is to see the speed difference between different volatility models. If the model is slow, it is not suitable for daily rebalancing.

## Setup

In [1]:
import os
import time
from pathlib import Path
from typing import get_args

import numpy as np
import pandas as pd

In [2]:
# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\couch\OneDrive\Assignments\Master Thesis\Repo


In [3]:
from src.config import VAL_END, WINDOW_SIZE
from src.models import VolatilityModel, predict_mean, predict_volatility

## Execution Timing

In [4]:
window_df = pd.read_parquet("results/processed_data/stock_data.parquet", engine="pyarrow")
window_df.index = pd.to_datetime(window_df.index)
window_df = window_df.loc[:VAL_END].tail(WINDOW_SIZE)
window_df

,ALR,CDR,CPS,DNP,EBP,JSW,KGH,LPP,LTS,MBK,MDV,OPL,PEO,PGE,PGN,PKN,PKO,PLY,PZU,TPE
Date,,,,,,,,,,,,,,,,,,,,
2017-12-21,0.005850,0.008004,0.009487,-0.015831,0.010317,-0.004263,0.007705,0.007232,-0.007843,-0.013993,0.003383,0.020034,-0.006939,-0.009102,-0.009917,0.028828,-0.003191,0.002682,-0.015812,-0.003284
2017-12-22,-0.002031,-0.001995,-0.000411,-0.008011,0.010974,-0.010263,-0.013637,-0.007903,-0.001542,-0.004184,-0.011263,-0.016667,-0.018743,-0.010865,0.014840,0.002703,-0.012637,-0.002981,-0.018970,-0.009917
2017-12-27,0.007595,0.018304,0.018312,0.013316,0.015114,-0.001053,0.022178,0.012658,0.011421,0.018692,0.020993,-0.016950,0.013313,0.005865,0.041673,0.016507,0.023987,0.006842,-0.011705,0.013202
2017-12-28,0.002519,-0.029853,0.012024,0.024822,-0.011187,0.019720,0.010243,-0.005865,-0.005098,-0.022894,-0.003527,-0.017242,0.011217,0.000000,-0.022223,-0.032377,0.009660,0.001481,0.031864,0.000000
2017-12-29,0.000000,-0.020409,-0.009608,0.016635,0.001768,-0.005387,-0.014729,-0.011161,-0.017183,-0.021277,0.007042,0.006932,-0.003854,0.006661,0.009585,-0.031572,-0.009434,0.000888,0.001662,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-19,-0.008316,0.002235,-0.035718,0.015839,-0.010724,-0.009950,0.010417,0.002861,-0.006439,0.011233,0.011215,-0.005450,0.003980,-0.011187,0.009479,0.012261,-0.007685,-0.011856,0.007701,-0.024244
2019-12-20,-0.015428,0.005197,0.014441,0.014185,0.000674,-0.009545,-0.015456,-0.003434,0.028654,-0.004799,-0.009337,-0.022100,-0.005976,-0.006270,-0.007101,-0.014637,-0.006881,0.017145,0.004444,-0.018576
2019-12-23,-0.009943,0.035997,0.002148,-0.004942,0.014706,0.010545,-0.000211,0.003434,0.017945,0.040843,0.007477,-0.015482,0.011423,0.002513,0.007101,0.012290,-0.002593,-0.012980,-0.006426,-0.006270


In [5]:
window_df.isna().sum()

ALR    0
CDR    0
CPS    0
DNP    0
EBP    0
JSW    0
KGH    0
LPP    0
LTS    0
MBK    0
MDV    0
OPL    0
PEO    0
PGE    0
PGN    0
PKN    0
PKO    0
PLY    0
PZU    0
TPE    0
dtype: int64

In [6]:
returns_matrix = np.asarray(window_df.values, dtype=np.float64)
timing_results = {}
volatility_models = list(get_args(VolatilityModel))

for model_name in volatility_models:
    t0 = time.perf_counter()
    try:
        _, demeaned_returns = predict_mean(returns_matrix, model="naive")
        cov_forecast, std_residuals = predict_volatility(demeaned_returns, model=model_name)
        elapsed = time.perf_counter() - t0
        timing_results[model_name.upper()] = elapsed
        print(f"[{model_name:>8}] Success | Time: {elapsed:.4f}s")
    except Exception as e:
        elapsed = time.perf_counter() - t0
        timing_results[model_name.upper()] = None
        print(f"[{model_name:>8}] Failed  | Time: {elapsed:.4f}s | Error: {e}")

# Summary DataFrame
benchmark_df = pd.DataFrame(list(timing_results.items()), columns=["Model", "Czas egzekucji (s)"])
benchmark_df = benchmark_df.set_index("Model")
benchmark_df.index.name = None
benchmark_df["Czas egzekucji (s)"] = benchmark_df["Czas egzekucji (s)"].apply(lambda x: f"{x:.4f}".replace(".", ","))
benchmark_df

[   naive] Success | Time: 0.0041s
[     ccc] Success | Time: 1.3548s
[     dcc] Success | Time: 7.5051s
[   dbekk] Success | Time: 809.9613s
[go_garch] Success | Time: 1.3135s


,Czas egzekucji (s)
NAIVE,"0,0041"
CCC,"1,3548"
DCC,"7,5051"
DBEKK,"809,9613"
GO_GARCH,"1,3135"
